In [1]:
import numpy as np 
import pandas as pd
import torch
from sklearn.metrics import r2_score
from pyro import clear_param_store
import pyro as pyro
import pyro.contrib.gp as gp
from pyro.nn import PyroSample
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS, Predictive,HMC
import arviz as az
from scipy.optimize import curve_fit


# Importing data 

In [2]:
PM25=pd.read_excel("../Pre_flow_cal.xlsx")
train,test=PM25.loc[PM25.label==0,:],PM25.loc[PM25.label==1,:]

FileNotFoundError: [Errno 2] No such file or directory: '../Pre_flow_cal.xlsx'

# Converting  PM25 data to tensors and put on GPU

In [5]:
# put data on cuda else cpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#make train and test data ready for flow of pump vs pm2.5 ready 
X25=torch.tensor(train.corrected_week.values).float().to(device)
y25=torch.tensor(train.flow.values).float().to(device)

In [7]:
clear_param_store();
rbf = gp.kernels.RBF(input_dim=1);

rbf.variance = PyroSample(dist.HalfNormal(torch.tensor(PM25.flow.mean())))

rbf.lengthscale = PyroSample(dist.HalfNormal(torch.tensor(15.)))
gpr = gp.models.GPRegression(X25,y25, rbf).to(device);
gpr.noise = PyroSample(dist.HalfNormal(PM25.flow.std()))

nuts_kernel = NUTS(gpr.model);

mcmc = MCMC(nuts_kernel, num_samples=7000,warmup_steps=7000,num_chains=1);

mcmc.run();

Sample: 100%|██████████████████████████████████████| 14000/14000 [04:18, 54.18it/s, step size=8.04e-01, acc. prob=0.906]    


In [11]:
torch.save(gpr, "../../Seasonality_functions/models/flow_season");

In [12]:
posterior_samples = mcmc.get_samples();
posterior_predictive = Predictive(gpr, posterior_samples)(X25);
prior = Predictive(gpr, num_samples=500)(X25);

pyro_data = az.from_pyro(mcmc,
    prior=prior,
    posterior_predictive=posterior_predictive

);
az.to_json(pyro_data, "../../Seasonality_functions/Arviz_stats/mcmc_season_flow.json")

Warmup:   1%|▎                                        | 70/10000 [14:15, 12.21s/it, step size=5.29e-02, acc. prob=0.778]
/data/michaelf/miniconda3/lib/python3.10/site-packages/arviz/data/io_pyro.py:158: UserWarning: Could not get vectorized trace, log_likelihood group will be omitted. Check your model vectorization or set log_likelihood=False
  warnings.warn(


'../../Seasonality_functions/Arviz_stats/mcmc_season_flow.json'